# 04 Neural Network Bigram


In [ ]:
import torch
import torch.nn.functional as F

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0

xs, ys = [], []
for w in words:
    chs = ["."] + list(w) + ["."]
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)
xenc = F.one_hot(xs, num_classes=27).float()

N = torch.zeros((27, 27), dtype=torch.int32)
for x, y in zip(xs, ys):
    N[x, y] += 1
P_count = (N + 1).float()
P_count /= P_count.sum(1, keepdim=True)
count_loss = -P_count[xs, ys].log().mean()

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

for k in range(200):
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    data_loss = -probs[torch.arange(len(ys)), ys].log().mean()
    loss = data_loss + 0.01 * (W**2).mean()

    W.grad = None
    loss.backward()
    W.data += -50 * W.grad

    if k % 20 == 0:
        print(k, loss.item())

print("count model:", count_loss.item())
print("neural network:", data_loss.item())
